In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.tsa.stattools import grangercausalitytests
from statsmodels.tsa.stattools import adfuller
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

merged = pd.read_csv("../data/processed/all_cities_combined.csv")
merged["week"] = pd.to_datetime(merged["week"])

CITIES = ["bangkok", "krakow", "melbourne", "lahore"]
TERMS  = ["air_quality", "air_purifier", "N95_mask"]
HEALTH_TERMS = ["hospital", "breathing_difficulty", "cough_medicine", "eye_drops", "chest_pain"]
ALL_TERMS = TERMS + HEALTH_TERMS
RF_CITIES = ["bangkok", "lahore"]
print("Ready ✓")
print(f"Shape: {merged.shape}")

In [ ]:
print("=== AUGMENTED DICKEY-FULLER STATIONARITY TEST ===")
print("(p < 0.05 = stationary = safe for Granger)\n")

for city in CITIES:
    df = merged[merged["city"] == city].dropna().copy()
    print(f"{city.upper()}")
    for col in ["pm25_mean", "air_quality"]:
        result = adfuller(df[col].dropna())
        p = result[1]
        status = "✓ stationary" if p < 0.05 else "✗ non-stationary — needs differencing"
        print(f"  {col:20s} → ADF p={p:.4f}  {status}")
    print()

In [ ]:
print("=== GRANGER CAUSALITY TEST ===")
print("Does PM2.5 Granger-cause search behavior?")
print("(p < 0.05 at any lag = PM2.5 has predictive power over search)\n")

MAX_LAG = 4
granger_p = {}

for city in CITIES:
    df = merged[merged["city"] == city].dropna().copy()
    df = df.sort_values("week").reset_index(drop=True)

    pm25 = df["pm25_mean"]
    search = df["air_quality"]

    adf_p = adfuller(search)[1]
    if adf_p >= 0.05:
        search = search.diff().dropna()
        pm25 = pm25.iloc[1:].reset_index(drop=True)
        note = "(differenced)"
    else:
        note = ""

    test_df = pd.DataFrame({
        "search": search.values,
        "pm25":   pm25.values
    }).dropna()

    print(f"{city.upper()} {note}")
    try:
        result = grangercausalitytests(test_df[["search", "pm25"]],
                                        maxlag=MAX_LAG, verbose=False)
        for lag in range(1, MAX_LAG + 1):
            f_stat = result[lag][0]["ssr_ftest"][0]
            p_val  = result[lag][0]["ssr_ftest"][1]
            sig = "✓ PM2.5 Granger-causes search" if p_val < 0.05 else "✗ no causal evidence"
            print(f"  Lag {lag}: F={f_stat:.3f}, p={p_val:.4f}  {sig}")
        granger_p[city] = result[MAX_LAG][0]["ssr_ftest"][1]
    except Exception as e:
        print(f"  ERROR: {e}")
    print()

In [ ]:
def build_features(city_df, target_term="air_quality", n_lags=4):
    df = city_df.sort_values("week").copy().reset_index(drop=True)

    for lag in range(1, n_lags + 1):
        df[f"pm25_lag{lag}"] = df["pm25_mean"].shift(lag)

    df["pm25_roll2"] = df["pm25_mean"].rolling(2).mean()
    df["pm25_roll4"] = df["pm25_mean"].rolling(4).mean()

    df["month_sin"] = np.sin(2 * np.pi * df["week"].dt.month / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["week"].dt.month / 12)

    feature_cols = (
        ["pm25_mean"] +
        [f"pm25_lag{i}" for i in range(1, n_lags+1)] +
        ["pm25_roll2", "pm25_roll4", "month_sin", "month_cos"]
    )

    df = df.dropna(subset=feature_cols + [target_term])
    X = df[feature_cols]
    y = df[target_term]

    return X, y, feature_cols

X, y, feat_cols = build_features(merged[merged["city"]=="bangkok"])
print(f"Feature matrix: {X.shape}")
print(f"Features: {feat_cols}")
print(f"\nSample:\n{X.head(3)}")

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error

print("=== RANDOM FOREST: PM2.5 → Search Index Prediction ===\n")

MIN_ROWS = 30
rf_results = {}

for city in CITIES:
    df = merged[merged["city"] == city].dropna().copy()
    city_terms = ALL_TERMS if city in RF_CITIES else ["air_quality"]
    print(f"{city.upper()}")

    for term in city_terms:
        try:
            X, y, feat_cols = build_features(df, target_term=term)

            if len(X) < MIN_ROWS:
                print(f"  {term:20s} → SKIPPED ({len(X)} rows, needs {MIN_ROWS})")
                continue

            if y.std() == 0:
                print(f"  {term:20s} → SKIPPED (constant series)")
                continue

            tscv = TimeSeriesSplit(n_splits=5)

            r2_scores, mae_scores = [], []

            for train_idx, test_idx in tscv.split(X):
                X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
                y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

                rf = RandomForestRegressor(
                    n_estimators=200,
                    max_depth=4,
                    min_samples_leaf=5,
                    random_state=42
                )
                rf.fit(X_train, y_train)
                y_pred = rf.predict(X_test)

                r2_scores.append(r2_score(y_test, y_pred))
                mae_scores.append(mean_absolute_error(y_test, y_pred))

            rf_final = RandomForestRegressor(n_estimators=200, max_depth=4,
                                              min_samples_leaf=5, random_state=42)
            rf_final.fit(X, y)

            rf_results[(city, term)] = {
                "r2_mean": np.mean(r2_scores),
                "r2_std":  np.std(r2_scores),
                "mae_mean": np.mean(mae_scores),
                "model": rf_final,
                "features": feat_cols,
                "X": X, "y": y
            }

            print(f"  {term:20s} → CV R²: {np.mean(r2_scores):.3f} ± {np.std(r2_scores):.3f} | "
                  f"CV MAE: {np.mean(mae_scores):.2f}")
        except Exception as e:
            print(f"  {term:20s} → ERROR: {e}")
    print()

In [ ]:
fig, axes = plt.subplots(1, len(CITIES), figsize=(5.3 * len(CITIES), 5))

feat_labels = {
    "pm25_mean":  "PM2.5 (current)",
    "pm25_lag1":  "PM2.5 lag 1w",
    "pm25_lag2":  "PM2.5 lag 2w",
    "pm25_lag3":  "PM2.5 lag 3w",
    "pm25_lag4":  "PM2.5 lag 4w",
    "pm25_roll2": "PM2.5 roll 2w",
    "pm25_roll4": "PM2.5 roll 4w",
    "month_sin":  "Seasonality (sin)",
    "month_cos":  "Seasonality (cos)"
}

for ax, city in zip(axes, CITIES):
    if (city, "air_quality") not in rf_results:
        ax.set_title(f"{city.capitalize()}\nnot modelled", fontweight="bold")
        ax.axis("off")
        continue
    res = rf_results[(city, "air_quality")]
    importances = res["model"].feature_importances_
    feat_names = [feat_labels[f] for f in res["features"]]

    sorted_idx = np.argsort(importances)

    colors = ["#d62728" if "current" in feat_names[i] or "lag" in feat_names[i] or "roll" in feat_names[i]
              else "#1f77b4" for i in sorted_idx]

    ax.barh([feat_names[i] for i in sorted_idx],
            importances[sorted_idx],
            color=[colors[i] for i in range(len(sorted_idx))],
            edgecolor="white")

    r2 = res["r2_mean"]
    ax.set_title(f"{city.capitalize()}\nCV R²={r2:.3f}", fontweight="bold")
    ax.set_xlabel("Feature Importance")
    ax.grid(axis="x", alpha=0.3)
    ax.set_xlim(0, 0.35)

plt.suptitle("Random Forest Feature Importances\nPredicting 'air quality' Search Index from PM2.5",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/figures/rf_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved → outputs/figures/rf_feature_importance.png")

In [ ]:
city = "bangkok"
res = rf_results[(city, "air_quality")]
X, y = res["X"], res["y"]

split = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

rf = RandomForestRegressor(n_estimators=200, max_depth=4,
                            min_samples_leaf=5, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

weeks = merged[merged["city"]==city].sort_values("week")["week"].iloc[4:].reset_index(drop=True)
test_weeks = weeks.iloc[split:]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test_weeks.values, y_test.values,
        color="#1f77b4", linewidth=2, label="Actual search index", marker="o", markersize=4)
ax.plot(test_weeks.values, y_pred,
        color="#d62728", linewidth=2, linestyle="--", label="Predicted (RF)", marker="s", markersize=4)
ax.fill_between(test_weeks.values, y_test.values, y_pred,
                alpha=0.15, color="grey")

ax.set_title(f"Bangkok — Actual vs Predicted Search Index\nRandom Forest | R²={r2:.3f}, MAE={mae:.2f}",
             fontweight="bold")
ax.set_ylabel("Search Index — 'air quality'")
ax.set_xlabel("Week")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("../outputs/figures/rf_predicted_vs_actual.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved → outputs/figures/rf_predicted_vs_actual.png")

In [ ]:
r2_matrix = np.full((len(CITIES), len(ALL_TERMS)), np.nan)

for i, city in enumerate(CITIES):
    for j, term in enumerate(ALL_TERMS):
        if (city, term) in rf_results:
            r2_matrix[i, j] = rf_results[(city, term)]["r2_mean"]

fig, ax = plt.subplots(figsize=(14, 5), layout="constrained")

ax.set_facecolor("#c8c8c8")
im = ax.imshow(r2_matrix, cmap="RdBu", vmin=-1, vmax=1, aspect="auto")

for i in range(len(CITIES)):
    for j in range(len(ALL_TERMS)):
        value = r2_matrix[i, j]
        if np.isnan(value):
            continue
        ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=10,
                color="white" if abs(value) > 0.6 else "black")

ax.set_xticks(range(len(ALL_TERMS)))
ax.set_xticklabels(ALL_TERMS, rotation=30, ha="right")
ax.set_yticks(range(len(CITIES)))
ax.set_yticklabels([city.capitalize() for city in CITIES])
ax.set_title("Random Forest CV R² — Cities × Search Terms (grey = not run, colour clipped to ±1)",
             fontsize=13, fontweight="bold")
fig.colorbar(im, ax=ax, label="CV R²")

plt.savefig("../outputs/figures/rf_r2_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved → outputs/figures/rf_r2_heatmap.png")

In [ ]:
print("=" * 92)
print(f"{'ML RESULTS SUMMARY':^92}")
print("=" * 92)

print(f"\n{'City':<12} {'Term':<22} {'CV R²':>8} {'CV MAE':>8} {'Top Feature':<20} {'Granger p@lag4':>15}")
print("-" * 92)

for (city, term), res in rf_results.items():
    top_idx = int(np.argmax(res["model"].feature_importances_))
    top_feature = feat_labels[res["features"][top_idx]]
    granger = f"{granger_p[city]:.4f}" if term == "air_quality" and city in granger_p else "-"
    print(f"{city.capitalize():<12} "
          f"{term:<22} "
          f"{res['r2_mean']:>8.3f} "
          f"{res['mae_mean']:>8.2f} "
          f"{top_feature:<20} "
          f"{granger:>15}")